# 🔒 Notebook 05 — Governance, Maintenance & Workflow Orchestration

1. **TBLPROPERTIES** — HIPAA classification, lineage tagging, retention
2. **GRANT/REVOKE** — role-based access control (requires UC Premium)
3. **Analyst views** — Gold-only, no member-level identifiers
4. **Delta OPTIMIZE + VACUUM** — maintenance across all 8 tables
5. **Databricks Workflow JSON** — 4-task DAG ready to import
6. **End-to-end lineage diagram**


## 1. Setup

In [0]:
import sys, os

# Auto-detect repo root — works for any user / workspace path
_nb_path  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/Workspace" + "/".join(_nb_path.split("/")[:-2])

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

CATALOG       = "medicare_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
ML_SCHEMA     = "ml_features"
VOLUME_PATH   = f"/Volumes/{CATALOG}/default/landing_zone/raw"

print(f"Repo root  : {repo_root}")
print(f"Catalog    : {CATALOG}")
print(f"Volume path: {VOLUME_PATH}")
_ok = os.path.exists(os.path.join(repo_root, "src"))
print("src/ found : ✅" if _ok else "❌ src/ missing — check repo clone")


## 2. TBLPROPERTIES — HIPAA lineage tagging
Tags appear in the Unity Catalog UI data lineage graph and in `DESCRIBE EXTENDED`.
They provide a machine-readable audit trail for every table.


In [0]:
TABLE_PROPS = {
    f"{CATALOG}.{BRONZE_SCHEMA}.raw_claims": {
        "data_classification": "HIPAA_deidentified",
        "pipeline_layer":      "bronze",
        "source":              "cms_synthetic_generator",
        "retention_days":      "2555",     # 7 years per HIPAA
        "owner":               "data_engineering",
    },
    f"{CATALOG}.{BRONZE_SCHEMA}.raw_members": {
        "data_classification": "HIPAA_deidentified",
        "pipeline_layer":      "bronze",
        "source":              "cms_synthetic_generator",
        "retention_days":      "2555",
    },
    f"{CATALOG}.{SILVER_SCHEMA}.clean_claims": {
        "data_classification": "HIPAA_deidentified",
        "pipeline_layer":      "silver",
        "derived_from":        f"{CATALOG}.{BRONZE_SCHEMA}.raw_claims",
        "quality_validated":   "true",
        "quality_flags":       "9",
    },
    f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims": {
        "data_classification": "HIPAA_deidentified",
        "pipeline_layer":      "silver",
        "hcc_version":         "v28",
        "derived_from":        f"{CATALOG}.{SILVER_SCHEMA}.clean_claims",
        "interaction_terms":   "CHF_AFib_0.175,CHF_DM_0.156,CKD_DM_0.156",
    },
    f"{CATALOG}.{GOLD_SCHEMA}.member_raf_scores": {
        "data_classification": "HIPAA_deidentified",
        "pipeline_layer":      "gold",
        "cms_model":           "HCC_v28",
        "derived_from":        f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims",
        "retention_days":      "2555",
    },
    f"{CATALOG}.{GOLD_SCHEMA}.monthly_trends": {
        "pipeline_layer":  "gold",
        "analysis_ready":  "DiD",
        "derived_from":    f"{CATALOG}.{SILVER_SCHEMA}.clean_claims",
    },
    f"{CATALOG}.{ML_SCHEMA}.risk_feature_store": {
        "pipeline_layer":  "ml_features",
        "ml_model":        "xgboost_risk_stratification",
        "feature_count":   "33",
        "leakage_guard":   "pre_period_only",
        "derived_from":    f"{CATALOG}.{GOLD_SCHEMA}.member_raf_scores",
    },
}

for table, props in TABLE_PROPS.items():
    prop_str = ", ".join([f"'{k}' = '{v}'" for k, v in props.items()])
    try:
        spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ({prop_str})")
        print(f"✅ {table}")
    except Exception as e:
        print(f"⚠️  Skipped {table.split('.')[-1]}: {str(e)[:80]}")


## 3. GRANT/REVOKE — role-based access control

**Access matrix** (mirrors real ACO data environment):
```
Role              Bronze    Silver    Gold      ML Features
────────────────  ───────   ───────   ───────   ───────────
analysts          ✗         ✗         SELECT    SELECT
engineers         ALL       ALL       SELECT    ✗
data scientists   ✗         SELECT    SELECT    ALL
auditors          SELECT    SELECT    SELECT    SELECT
```
**Why analysts cannot see Silver**: Silver contains claim-line detail with `bene_id`
and ICD-10 codes. HIPAA minimum-necessary principle — analysts consume pre-aggregated
Gold views and have no clinical need for individual claim lines.

Note: GRANT/REVOKE requires Unity Catalog Premium workspace. Skipped gracefully on Community Edition.


In [0]:
_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

# Update group names to match your workspace groups
GROUPS = {
    "analysts":        f"analysts",
    "engineers":       f"engineers",
    "data_scientists": f"data_scientists",
    "auditors":        f"auditors",
}

GRANTS = [
    # Analysts — Gold + ML (read-only)
    f"GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `{GROUPS['analysts']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{ML_SCHEMA}   TO `{GROUPS['analysts']}`",
    # Engineers — full Bronze + Silver
    f"GRANT ALL PRIVILEGES ON SCHEMA {CATALOG}.{BRONZE_SCHEMA} TO `{GROUPS['engineers']}`",
    f"GRANT ALL PRIVILEGES ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `{GROUPS['engineers']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `{GROUPS['engineers']}`",
    # Data scientists — ML full, Silver + Gold read
    f"GRANT ALL PRIVILEGES ON SCHEMA {CATALOG}.{ML_SCHEMA}     TO `{GROUPS['data_scientists']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `{GROUPS['data_scientists']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA}   TO `{GROUPS['data_scientists']}`",
    # Auditors — read everything
    f"GRANT SELECT ON SCHEMA {CATALOG}.{BRONZE_SCHEMA} TO `{GROUPS['auditors']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `{GROUPS['auditors']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA}   TO `{GROUPS['auditors']}`",
    f"GRANT SELECT ON SCHEMA {CATALOG}.{ML_SCHEMA}     TO `{GROUPS['auditors']}`",
]

for stmt in GRANTS:
    try:
        spark.sql(stmt)
        print(f"✅ {stmt[6:60]}...")
    except Exception as e:
        print(f"⚠️  Skipped (update group names or use UC Premium): {str(e)[:100]}")


## 4. Analyst-facing Gold views — no member-level identifiers

In [0]:
# SQL built with string concatenation to avoid nested triple-quote conflicts
_v_risk = (
    "SELECT risk_tier, COUNT(*) AS n_members, "
    "ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER (), 1) AS pct_of_population, "
    "ROUND(AVG(raf_score),3) AS mean_raf, "
    "ROUND(AVG(actual_pmpm),2) AS mean_pmpm_usd, "
    "ROUND(AVG(estimated_annual_cost),0) AS mean_est_cost_usd, "
    "SUM(ip_admit_count) AS total_ip_admits, SUM(ed_visit_count) AS total_ed_visits "
    f"FROM {CATALOG}.{GOLD_SCHEMA}.member_raf_scores GROUP BY risk_tier"
)
_v_trends = (
    "SELECT claim_year, claim_month, intervention_arm, n_members, "
    "ROUND(pmpm_cost,2) AS pmpm_usd, ROUND(pmpm_rolling_3m,2) AS pmpm_3m_usd, "
    "ROUND(ip_rate_per_1000,1) AS ip_per_1000, ROUND(ed_rate_per_1000,1) AS ed_per_1000, period "
    f"FROM {CATALOG}.{GOLD_SCHEMA}.monthly_trends "
    "ORDER BY claim_year, claim_month, intervention_arm"
)
_v_hcc = (
    "SELECT primary_hcc_desc AS condition, primary_hcc AS hcc_number, "
    "COUNT(DISTINCT bene_id) AS n_members, COUNT(claim_id) AS n_claims, "
    "ROUND(AVG(hcc_raf_total),3) AS mean_hcc_raf, ROUND(SUM(claim_amount),0) AS total_cost_usd "
    f"FROM {CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims "
    "WHERE primary_hcc_desc IS NOT NULL AND primary_hcc_desc != 'Not Mapped' "
    "AND _quality_pass = true "
    "GROUP BY primary_hcc_desc, primary_hcc ORDER BY n_members DESC LIMIT 20"
)

VIEWS = {
    f"{CATALOG}.{GOLD_SCHEMA}.v_high_risk_summary": _v_risk,
    f"{CATALOG}.{GOLD_SCHEMA}.v_cohort_trends":      _v_trends,
    f"{CATALOG}.{GOLD_SCHEMA}.v_top_hcc_conditions": _v_hcc,
}

for view_name, view_sql in VIEWS.items():
    try:
        spark.sql(f"CREATE OR REPLACE VIEW {view_name} AS {view_sql}")
        print(f"✅ {view_name}")
    except Exception as e:
        print(f"⚠️  {view_name.split('.')[-1]}: {str(e)[:100]}")


In [0]:
print("=== v_high_risk_summary ===")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.v_high_risk_summary"))


In [0]:
print("=== v_top_hcc_conditions ===")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.v_top_hcc_conditions"))


## 5. Delta OPTIMIZE + VACUUM — all tables

In [0]:
ALL_TABLES = [
    f"{CATALOG}.{BRONZE_SCHEMA}.raw_claims",
    f"{CATALOG}.{BRONZE_SCHEMA}.raw_members",
    f"{CATALOG}.{SILVER_SCHEMA}.clean_claims",
    f"{CATALOG}.{SILVER_SCHEMA}.hcc_mapped_claims",
    f"{CATALOG}.{SILVER_SCHEMA}.member_profile",
    f"{CATALOG}.{GOLD_SCHEMA}.member_raf_scores",
    f"{CATALOG}.{GOLD_SCHEMA}.utilization_summary",
    f"{CATALOG}.{GOLD_SCHEMA}.monthly_trends",
    f"{CATALOG}.{ML_SCHEMA}.risk_feature_store",
]

for t in ALL_TABLES:
    try:
        spark.sql(f"OPTIMIZE {t}")
        print(f"✅ OPTIMIZE: {t.split('.')[-1]}")
    except Exception as e:
        print(f"⚠️  {t.split('.')[-1]}: {str(e)[:80]}")


In [0]:
for t in ALL_TABLES:
    try:
        spark.sql(f"VACUUM {t} RETAIN 168 HOURS")  # 7 days
        print(f"✅ VACUUM: {t.split('.')[-1]}")
    except Exception as e:
        print(f"⚠️  {t.split('.')[-1]}: {str(e)[:80]}")


## 6. Databricks Workflow — 4-task DAG

In [0]:
import json

_user   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_base = "/".join(_nb_path.split("/")[:-1])  # parent dir of this notebook

workflow = {
    "name": "medicare_lakehouse_daily",
    "description": "Medicare Claims Lakehouse: Bronze → Silver → Gold → Model Retrain",
    "schedule": {
        "quartz_cron_expression": "0 0 2 * * ?",
        "timezone_id": "UTC",
        "pause_status": "UNPAUSED"
    },
    "tasks": [
        {
            "task_key": "bronze_ingestion",
            "description": "Generate synthetic data + ingest to Bronze Delta",
            "notebook_task": {
                "notebook_path": f"{nb_base}/01_bronze_ingestion",
                "base_parameters": {"n_members": "10000", "n_months": "24", "batch_id": ""}
            },
            "existing_cluster_id": "REPLACE_WITH_YOUR_CLUSTER_ID",
            "timeout_seconds": 1800,
            "max_retries": 2,
            "retry_on_timeout": True,
        },
        {
            "task_key": "silver_processing",
            "description": "Dedup + quality-flag + HCC-map → Silver Delta",
            "depends_on": [{"task_key": "bronze_ingestion"}],
            "notebook_task": {"notebook_path": f"{nb_base}/02_silver_processing"},
            "existing_cluster_id": "REPLACE_WITH_YOUR_CLUSTER_ID",
            "timeout_seconds": 2400,
            "max_retries": 1,
        },
        {
            "task_key": "gold_aggregation",
            "description": "RAF scores + feature store → Gold Delta",
            "depends_on": [{"task_key": "silver_processing"}],
            "notebook_task": {"notebook_path": f"{nb_base}/03_gold_aggregates"},
            "existing_cluster_id": "REPLACE_WITH_YOUR_CLUSTER_ID",
            "timeout_seconds": 2400,
            "max_retries": 1,
        },
        {
            "task_key": "model_retrain",
            "description": "Train XGBoost + SHAP + PSI drift → MLflow Registry",
            "depends_on": [{"task_key": "gold_aggregation"}],
            "notebook_task": {"notebook_path": f"{nb_base}/04_mlflow_model"},
            "existing_cluster_id": "REPLACE_WITH_YOUR_CLUSTER_ID",
            "timeout_seconds": 3600,
            "max_retries": 1,
        },
    ],
    "email_notifications": {
        "on_failure": [_user],
        "no_alert_for_skipped_runs": True,
    },
    "tags": {
        "project": "medicare_lakehouse",
        "environment": "production",
        "hipaa": "deidentified_synthetic",
    }
}

print("=== Paste into: Workflows → Create Job → Edit JSON ===")
print()
print(json.dumps(workflow, indent=2))


## 7. End-to-end lineage

```
CMS Synthetic Generator (pandas, driver)
        │
        ▼  [generate_and_save()]
UC Volume: /Volumes/medicare_lakehouse/default/landing_zone/raw/
  ├── members.csv   (~2 MB)
  └── claims.csv    (~300 MB)
        │
        ▼  [StructType schema enforcement + audit columns + Bronze null flag]
        │   Single lazy DAG — one write action
bronze.raw_claims     (partitioned claim_year/month, ZORDER bene_id)
bronze.raw_members    (full snapshot)
        │
        ▼  [dedup ROW_NUMBER + 9 quality flags + pandas_udf HCC closures]
        │   No spark.sparkContext — Serverless-native
silver.clean_claims          (_quality_pass, 9 flags, service features)
silver.hcc_mapped_claims     (hcc_list, raf_total, interaction terms, 8 condition flags)
silver.member_profile        (demographic_raf, age_bracket)
        │
        ▼  [member-level aggregation + no-leakage feature engineering]
gold.member_raf_scores        (raf_score, risk_tier, pmpm — primary analytics output)
gold.utilization_summary      (IP/ED/PMPM by member × period)
gold.monthly_trends           (DiD-ready: cohort × month × arm, 3M rolling avg)
ml_features.risk_feature_store (33 features, pre-period only, log-transformed)
        │
        ▼  [XGBoost + isotonic calibration + SHAP on driver, MLflow Serverless]
MLflow Experiment: medicare_raf_risk_stratification
  └── Model Registry: medicare_raf_risk_model → Staging → Production
        │
        ▼  [Unity Catalog views — no bene_id exposure]
gold.v_high_risk_summary       (tier rollup: RAF, PMPM, IP/ED)
gold.v_cohort_trends           (monthly trends, rolling avg)
gold.v_top_hcc_conditions      (condition prevalence, RAF, cost)
        │
        ▼  [Databricks Workflows — 4-task DAG, daily 02:00 UTC]
bronze_ingestion → silver_processing → gold_aggregation → model_retrain
```


## ✅ Pipeline fully operational

| Component | Notes |
|-----------|-------|
| Bronze Delta | ✅ Partitioned, ZORDER, audit columns, time-travel |
| Silver HCC mapping | ✅ pandas_udf closures — Serverless-native, no sparkContext |
| Gold + Feature store | ✅ RAF scores, DiD trends, 33-feature ML matrix, zero leakage |
| MLflow experiment | ✅ Params, metrics, SHAP plot, classification report |
| Model Registry | ✅ Versioned, Staging promotion |
| PSI drift monitoring | ✅ 8 features, structured PSIReport |
| Unity Catalog | ✅ TBLPROPERTIES lineage, RBAC views |
| Databricks Workflow | ✅ 4-task DAG with retries, email alerting |
